#### Processed the csv file

In [1]:
from langchain_community.document_loaders import CSVLoader

FILE = "./documents/survey.csv"

loader = CSVLoader(file_path=FILE)
docs = loader.load()

C:\Users\rahul\AppData\Local\Temp\ipykernel_5492\4009486548.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [2]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents=docs)

In [6]:
import os
from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from pydantic import SecretStr


load_dotenv(override=True)

BASE_URL: str = os.getenv("OLLAMA_BASE_URL", "")
EMBEDDING_MODEL: str = os.getenv("OLLAMA_EMBEDDING_MODEL", "")

OPENAI_API_KEY: SecretStr = SecretStr(os.getenv("OPENAI_API_KEY", ""))
OPENAI_BASE_URL: str = os.getenv("OPENAI_BASE_URL", "")

embedding = OllamaEmbeddings(base_url=BASE_URL, model=EMBEDDING_MODEL)
LLM = ChatOpenAI(
    model="nvidia/nemotron-3-nano-4b",
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
)

In [7]:
from langchain_chroma import Chroma

db = Chroma.from_documents(documents=docs, embedding=embedding)  # type: ignore

In [ ]:
from langchain_classic.prompts import PromptTemplate


template = """You are a clothing consultant chatbot. 
Answer the customer questions only using source data provided. Please answer to their specific question. 
If you are unsure, say 'I don't know, please call our customer support'. Use engaging, courages reply.
Keep your answers concise.
{context}
"""

prompt = PromptTemplate(template=template, input_variables=["context"])

In [9]:
from langchain_classic.chains import RetrievalQA

chain_type_kwargs = {"prompt": prompt}

chain = RetrievalQA.from_chain_type(
    llm=LLM,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 1}),
    chain_type_kwargs=chain_type_kwargs
)

In [16]:
query = "What size do you carry?"

response = chain.invoke(query)
print(response["result"])


We’ve got everything from XS to XXL, plus a few styles that go even larger! 🌟


In [17]:
query = "Is their a way I can get the stock details? like which item is in stock?"

response = chain.invoke(query)
print(response["result"])


Check out our site—it’s updated in real‑time so you’ll see exactly what’s in stock. Want a heads‑up when it’s back? Sign up for restock notifications!
